## Аминокислотные последовательности антител

<img src="https://i0.wp.com/www.blopig.com/blog/wp-content/uploads/2013/07/Antibody1.png?ssl=1" style="background:white" width="600"/>

Как и любые белки́, [антитела](https://ru.wikipedia.org/wiki/%D0%90%D0%BD%D1%82%D0%B8%D1%82%D0%B5%D0%BB%D0%B0) состоят из [аминокислот](https://shorturl.at/Wc6V4), соединённых пептидными связями.

Каждой аминокислоте соответствует однобуквенный код (A - аланин, C - цистеин, D - аспартат, и т.д.), поэтому мы можем компактно записывать первичную структуру белков (то есть просто цепочку аминокислот) в виде строк.

Сегодня нам предстоит решить две задачи:
1. Обучить модель-классификатор, позволяющую по аминокислотной последовательности понять, из какого животного было получено антитело
2. Обучить модель-генератор новых антител с условием на биологический вид

#### Подготовим данные

В нашем наборе данных присутствуют антитела (точнее - небольшие фрагменты тяжелых цепей антител, VH-домены, непосредственно участвующие в связывании инородных молекул - антигенов), полученные из образцов пяти биологических видов: человек, макака-резус, мышь, кролик и верблюд.

In [1]:
import pandas as pd

antibodies = pd.read_csv("antibodies.csv").sample(frac=1.0)
antibodies.head(10)

,sequence,species
3864,EVHLVESGGGLAEPGGSLSLSCAASGITFRTYWRHWIRRAPGKGLE...,Rhesus
2535,SVKVSCKASGGTFSSYAISWLRQAPGQGPEWMGGIIPIFNAADYTQ...,Mouse
3110,EVQLGESGGGLAKPGGSLRLSCAASGFIFGDHYMPWVRQASGKGLE...,Rhesus
4437,QEQLVESGGGLVKPGASLTLTCKASGFPFSDMAVICWVRQAPGKGL...,Rabbit
1252,HVQLVESGGGSVQAGGSLTLFCEISKADDSTVCMAWFRQAPGKERE...,Camel
2112,SVKVSCKASGYTFTGYYLHWVRQSPRLGLEWLGRINPNTGDTDYSK...,Mouse
1951,HVQLVESGGGLVQAGGSLNLSCAATGKTNVLNCMGWFRQAPGKDRE...,Camel
2554,SVKVSCKASGYTFTSYYMHWVRQAPGQGLEWMGIINPSGGSTSYAQ...,Mouse
4889,QEQLVESGGGLVKPEGSLKLSCTASGFSFSNKAVMCWVRQTPGKGL...,Rabbit
1478,QVQLVESGGGSVQVGGSLNLTCAVSGDTHSSLCMGWFRQSPETEPE...,Camel


Создадим словари для аминокислот, специальных токенов и биологических видов:

In [2]:
SPECIAL_TOKENS = "_?\n"
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWYX"
VOCAB = {char: i for i, char in enumerate(SPECIAL_TOKENS + AMINO_ACIDS)}
SPECIES = {name: i for i, name in enumerate(sorted(antibodies["species"].unique()))}
SPECIES

{'Camel': 0, 'Human': 1, 'Mouse': 2, 'Rabbit': 3, 'Rhesus': 4}

In [3]:
VOCAB

{'_': 0,
 '?': 1,
 '\n': 2,
 'A': 3,
 'C': 4,
 'D': 5,
 'E': 6,
 'F': 7,
 'G': 8,
 'H': 9,
 'I': 10,
 'K': 11,
 'L': 12,
 'M': 13,
 'N': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'V': 20,
 'W': 21,
 'Y': 22,
 'X': 23}

In [4]:
ANTIVOCAB = {i: char for i, char in enumerate(VOCAB)}
SPECIAL_NUMBERS = list(map(lambda x: VOCAB[x], SPECIAL_TOKENS))
SPECIAL_NUMBERS, ANTIVOCAB

([0, 1, 2],
 {0: '_',
  1: '?',
  2: '\n',
  3: 'A',
  4: 'C',
  5: 'D',
  6: 'E',
  7: 'F',
  8: 'G',
  9: 'H',
  10: 'I',
  11: 'K',
  12: 'L',
  13: 'M',
  14: 'N',
  15: 'P',
  16: 'Q',
  17: 'R',
  18: 'S',
  19: 'T',
  20: 'V',
  21: 'W',
  22: 'Y',
  23: 'X'})

In [5]:
def decode_tokens(encoded: list[int], remove_special_tokens: bool = False) -> str:
        return ''.join(
            [
                ANTIVOCAB[idx]
                for idx in encoded
                if (idx not in SPECIAL_NUMBERS) or not remove_special_tokens
            ]
        )

In [6]:
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, random_split


class AntibodiesDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.sequences = [
            [1] + [VOCAB[char] for char in s] + [2] for s in df["sequence"]
        ]
        self.labels = [SPECIES[label] for label in df["species"]]

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.sequences[index], self.labels[index]

    def __len__(self) -> int:
        return len(self.sequences)

    @staticmethod
    def collate_fn(batch: list[tuple[list[int], int]]) -> tuple[Tensor, Tensor]:
        encoded, lang_ids = zip(*batch)
        max_len = max(map(len, encoded))
        x = torch.zeros((len(encoded), max_len), dtype=int)
        for i, seq in enumerate(encoded):
            x[i, : len(seq)] = torch.tensor(seq)

        return x, torch.tensor(list(lang_ids))


labels = sorted(antibodies["species"].unique())
dataset = AntibodiesDataset(antibodies)
train_dataset, test_dataset = random_split(
    dataset, [4500, 500], torch.Generator().manual_seed(42)
)
print("Train size: ", len(train_dataset))
print("Test size: ", len(test_dataset))

Train size:  4500
Test size:  500


In [7]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=dataset.collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, collate_fn=dataset.collate_fn
)

#### Задание 1 (6 баллов). Классификация антител по биологическим видам

Мы начнём с нашей рекуррентной ячейки с последней практики:

In [8]:
class RNNCell(nn.Module):
    """
    (x_{t}, h_{t-1}) -> h_{t}
    """

    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.linear = nn.Linear(input_dim + hidden_dim, hidden_dim)

    def forward(self, x: Tensor, h: Tensor) -> Tensor:
        # x: B x input_dim
        # h: B x hidden_dim
        h = torch.cat([x, h], dim=1)
        h = self.linear(h)
        return F.tanh(h)

class LSTMCell(nn.Module):
    """
    (x_{t}, h_{t-1}) -> h_{t}
    """

    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.linear_f = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_i = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_c = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_o = nn.Linear(input_dim + hidden_dim, hidden_dim)

    def forward(self, x: Tensor, h: Tensor, C: Tensor) -> Tensor:
        f = torch.sigmoid(self.linear_f(torch.cat([x, h], dim=1)))
        i = torch.sigmoid(self.linear_i(torch.cat([x, h], dim=1)))
        C_ = F.tanh(self.linear_c(torch.cat([x, h], dim=1)))
        C = f * C + i * C_
        o = torch.sigmoid(self.linear_o(torch.cat([x, h], dim=1)))
        return o * F.tanh(C), C

**1.1. (1 балл)** Реализуйте архитектуру модели, которая по входной последовательности будет давать вероятностное распределение над биологическими видами.
Она даже немного проще, чем модель для генерации: линейный блок-классификатор мы применяем только к последнему скрытому состоянию (когда вся последовательность обработана).

**1.2. (2 балла)** Обучите модель в течение 10-50 эпох, постройте графики точности классифкации для обучающей и тестовой выборок.

**1.3. (3 балла)** Реализуйте другой вид рекуррентной ячейки (GRU или LSTM, см. практику), обучите модель на его основе, выведите графики точности. Как изменилась точность модели и скорость обучения?


Указание: используйте небольшие модели, с размером скрытого слоя 64

In [9]:
import torch.nn as nn
from typing import Literal

class RNN(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_dim: int = 64, n_classes: int = 5,
        is_lstm: bool = False
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.init_h = nn.Parameter(data=torch.randn(1, hidden_dim))
        self.rnn = RNNCell(hidden_dim, hidden_dim)
        self.lstm = LSTMCell(hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, n_classes)
        self.init_C = nn.Parameter(data=torch.randn(1, hidden_dim))
        self.is_lstm = is_lstm

    def forward(self, x: Tensor) -> Tensor:
        # x: B x T
        # embed(x): B x T -> B x T x hidden_dim
        B, T = x.shape

        x = self.embed(x)  # B x T x hidden_dim
        h = self.init_h.expand((B, -1)) # B x hidden_dim

        C = self.init_C.expand((B, -1))

        for t in range(T):
            xt = x[:, t, :]
            if self.is_lstm:
                h, C = self.lstm.forward(xt, h, C)
            else:
                h = self.rnn.forward(xt, h)

        y = self.lm_head(h)  # B x n_classes
        return y

In [10]:
tokens, labels = next(iter(train_loader))
print("Tokens shape: ", tokens.shape, "\nLabels shape: ", labels.shape)
# print(tokens)
# print(labels)
model = RNN(
    vocab_size=len(VOCAB),
)
torch.nn.CrossEntropyLoss()(model.forward(tokens), labels)
# no errors, nice

Tokens shape:  torch.Size([32, 129]) 
Labels shape:  torch.Size([32])


tensor(1.6162, grad_fn=<NllLossBackward0>)

In [11]:
from collections import defaultdict
from typing import Callable

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def training_step(
    batch: tuple[torch.Tensor, torch.Tensor],
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: str = "cpu",
) -> tuple[Tensor, Tensor]:
    model.to(device=device)
    # прогоняем батч через модель
    x, y = batch
    logits = model(x.to(device=device))
    # оцениваем значение ошибки
    loss = F.cross_entropy(logits, y.to(device=device))
    # обновляем параметры
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    # возвращаем значение функции ошибки для логирования
    return loss, logits


def train_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    max_batches: int = 100,
    device: str = "cpu",
) -> dict[str, float]:
    model.train()
    loss_total = 0
    n_correct = 0
    n_total = 0
    for i, batch in enumerate(dataloader):
        x, y = batch
        loss, logits = training_step(batch, model, optimizer, device)

        # save stats
        n_total += y.size(0)
        n_correct += (y.to(device=device) == logits.argmax(dim=1)).sum()
        loss_total += y.size(0) * loss.item()
        if i == max_batches:
            break

    return {
        "loss": loss_total / n_total,
        "accuracy": n_correct / n_total,
    }


@torch.no_grad()
def test_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    max_batches: int = 100,
    device: str = "cpu",
) -> Tensor:
    model.eval()
    model.to(device=device)
    loss_total = 0
    n_correct = 0
    n_total = 0
    for i, batch in enumerate(dataloader):
        x, y = batch
        logits = model(x.to(device=device))
        # оцениваем значение ошибки
        loss = F.cross_entropy(logits, y.to(device=device))
        # save stats
        n_total += y.size(0)
        n_correct += (y.to(device=device) == logits.argmax(dim=1)).sum()
        loss_total += y.size(0) * loss.item()
        if i == max_batches:
            break

    return {
        "loss": loss_total / n_total,
        "accuracy": n_correct / n_total,
    }


def run_experiment(
    model_gen: Callable[[], nn.Module],
    optim_gen: Callable[[nn.Module], torch.optim.Optimizer],
    seed: int,
    n_epochs: int = 10,
    max_batches: int | None = None,
    verbose: bool = False,
    device: str = "cpu",
) -> dict[str, list[float]]:
    torch.manual_seed(seed)
    model = model_gen()
    optim = optim_gen(model)
    metrics: dict[str, list[float]] = defaultdict(list)
    for i in range(n_epochs):
        train_dict = train_epoch(
            train_loader, model, optim, max_batches=max_batches, device=device
        )
        test_dict = test_epoch(
            test_loader, model, max_batches=max_batches, device=device
        )
        train_loss, train_accuracy = train_dict["loss"], train_dict["accuracy"]
        test_loss, test_accuracy = test_dict["loss"], test_dict["accuracy"]
        if verbose:
            print(
                f"Epoch {i} train: loss = {train_loss:.4f}, accuracy = {train_accuracy:.4f}"
            )
            print(
                f"Epoch {i} test: loss = {test_loss:.4f}, accuracy = {test_accuracy:.4f}"
            )

        metrics["train_losses"].append(train_loss)
        metrics["train_accuracies"].append(train_accuracy)
        metrics["test_losses"].append(test_loss)
        metrics["test_accuracies"].append(test_accuracy)

    return metrics

In [12]:
from time import perf_counter
print("no lstm")

start=perf_counter()
metrics_dict = run_experiment(
    model_gen=lambda: RNN(vocab_size=len(VOCAB), is_lstm=False),
    optim_gen=lambda x: torch.optim.Adam(x.parameters(), lr=0.001),
    seed=42,
    n_epochs=10,
    max_batches=None,
    verbose=True,
    device="cuda",
)
print(f"ran {perf_counter()-start} seconds")

no lstm
Epoch 0 train: loss = 1.5505, accuracy = 0.2718
Epoch 0 test: loss = 1.3348, accuracy = 0.4300
Epoch 1 train: loss = 1.4965, accuracy = 0.3356
Epoch 1 test: loss = 1.5245, accuracy = 0.3020
Epoch 2 train: loss = 1.5130, accuracy = 0.3009
Epoch 2 test: loss = 1.5540, accuracy = 0.2660
Epoch 3 train: loss = 1.4714, accuracy = 0.3513
Epoch 3 test: loss = 1.4202, accuracy = 0.3700
Epoch 4 train: loss = 1.3122, accuracy = 0.4120
Epoch 4 test: loss = 1.2048, accuracy = 0.4540
Epoch 5 train: loss = 1.2798, accuracy = 0.4296
Epoch 5 test: loss = 1.2167, accuracy = 0.4520
Epoch 6 train: loss = 1.2570, accuracy = 0.4347
Epoch 6 test: loss = 1.3238, accuracy = 0.4080
Epoch 7 train: loss = 1.2637, accuracy = 0.4382
Epoch 7 test: loss = 1.1729, accuracy = 0.4540
Epoch 8 train: loss = 1.2459, accuracy = 0.4322
Epoch 8 test: loss = 1.1862, accuracy = 0.4440
Epoch 9 train: loss = 1.2609, accuracy = 0.4322
Epoch 9 test: loss = 1.1963, accuracy = 0.4400
ran 69.645454288 seconds


In [13]:
from time import perf_counter
print("lstm")

start=perf_counter()
metrics_dict = run_experiment(
    model_gen=lambda: RNN(vocab_size=len(VOCAB), is_lstm=True),
    optim_gen=lambda x: torch.optim.Adam(x.parameters(), lr=0.001),
    seed=42,
    n_epochs=10,
    max_batches=None,
    verbose=True,
    device="cuda",
)
print(f"ran {perf_counter()-start} seconds")

lstm
Epoch 0 train: loss = 1.3713, accuracy = 0.3818
Epoch 0 test: loss = 1.1207, accuracy = 0.5080
Epoch 1 train: loss = 0.8308, accuracy = 0.6831
Epoch 1 test: loss = 0.4475, accuracy = 0.8640
Epoch 2 train: loss = 0.3028, accuracy = 0.9042
Epoch 2 test: loss = 0.2003, accuracy = 0.9440
Epoch 3 train: loss = 0.1600, accuracy = 0.9553
Epoch 3 test: loss = 0.0935, accuracy = 0.9780
Epoch 4 train: loss = 0.1031, accuracy = 0.9691
Epoch 4 test: loss = 0.0676, accuracy = 0.9820
Epoch 5 train: loss = 0.0716, accuracy = 0.9802
Epoch 5 test: loss = 0.0648, accuracy = 0.9860
Epoch 6 train: loss = 0.0652, accuracy = 0.9807
Epoch 6 test: loss = 0.0599, accuracy = 0.9840
Epoch 7 train: loss = 0.0754, accuracy = 0.9782
Epoch 7 test: loss = 0.0642, accuracy = 0.9820
Epoch 8 train: loss = 0.0423, accuracy = 0.9876
Epoch 8 test: loss = 0.0757, accuracy = 0.9700
Epoch 9 train: loss = 0.0387, accuracy = 0.9876
Epoch 9 test: loss = 0.0333, accuracy = 0.9920
ran 234.056151601 seconds


Версия с LSTM работает на порядок дольше, но как в loss, так и в accuracy мгновенно происходит скачок. Наверное, дело в том, что модель научилась запоминать характерные буквы на характерных позициях.

#### Задание 2 (8 баллов + 4 бонусных). Генерация антител

Поиграем за B-лимфоцит и попробуем создать новые антитела.

Модель - почти полная копия модели с практики, но есть дополнительное условие: теперь кроме текущего токена и предыдущего скрытого состояния пусть наша ячейка принимает ещё метку биологического вида, к которому должно относиться антитело, вроде такого:

In [14]:
class ConditionalRNNCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        # replace 2nd hidden_dim with C_dim?
        self.linear = nn.Linear(input_dim + hidden_dim + hidden_dim, hidden_dim)

    """
    (x_{t}, h_{t-1}, c) -> h_{t}
    """
    def forward(self, x: Tensor, c: Tensor, h: Tensor) -> Tensor:
        # x: B x input_dim: эмбеддинг последнего токен
        # c: B x class_dim: эмбеддинг биологического вида
        # h: B x hidden_dim: последнее скрытое состояние
        h = torch.cat([x, h, c], dim=1)
        h = self.linear(h)
        return F.tanh(h)


import torch.nn as nn
from typing import Literal

class RNNGen(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_dim: int = 64, n_classes: int = 5
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.embed_c = nn.Embedding(n_classes, hidden_dim)
        self.init_h = nn.Parameter(data=torch.randn(1, hidden_dim))
        self.rnn = ConditionalRNNCell(hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: Tensor, c: Tensor) -> Tensor:
        # x: B x T
        # embed(x): B x T -> B x T x hidden_dim
        B, T = x.shape

        x = self.embed(x)  # B x T x hidden_dim
        c = self.embed_c(c).expand((B, -1))  # B x hidden_dim
        h = self.init_h.expand((B, -1)) # B x hidden_dim

        logits = []
        for t in range(T):
            xt = x[:, t, :] # B x hidden
            h = self.rnn.forward(xt, c, h)  # B x hidden
            y = self.lm_head(h).unsqueeze(1)  # B x 1 x hidden
            logits.append(y)

        return torch.cat(logits, dim=1)

tokens, labels = next(iter(train_loader))
print("Tokens shape: ", tokens.shape, "\nLabels shape: ", labels.shape)
# print(tokens)
# print(labels)
model = RNNGen(
    vocab_size=len(VOCAB),
)
print("vocab", len(VOCAB))
logits = model.forward(tokens, labels)
print("Logits shape: ", logits.shape)
# torch.nn.CrossEntropyLoss()(model.forward(tokens, labels), )
# no errors, nice

Tokens shape:  torch.Size([32, 131]) 
Labels shape:  torch.Size([32])
vocab 24
Logits shape:  torch.Size([32, 131, 24])


**2.1. (2 балла)** Реализуйте архитектуру модели для генерации антител символ за символом

**2.2. (2 балла)** Обучите модель в течение 10-50 эпох, постройте графики функции ошибки.

**2.3. (4 балла)** Сгенерируйте с помощью модели по 20 антител для каждого биологического вида в отдельных ячейках, выведите их в ноутбуке. Воспользуйтесь функцией `get_sequence_score` (ниже), чтобы посчитать сходство ваших антител с природными (вернее - очень грубую оценку). Посчитайте, сколько антител из сгенерированных вами имеют оценку 0.55 и выше для каждого биологического вида.

**2.4. (Бонус 4 балла)** Повторите пункты 2.2 и 2.3, но используйте другой тип рекуррентной ячейки (GRU или LSTM).


In [15]:
# print(len(VOCAB))
F.cross_entropy(
    logits[:, :-1].reshape(-1, len(VOCAB)),
    tokens[:, 1:].reshape(-1),
)

tensor(3.2328, grad_fn=<NllLossBackward0>)

In [16]:
@torch.no_grad()
def generate(model: RNNGen, idx: Tensor, c: Tensor, max_new_tokens: int) -> Tensor:
    # idx: B x T
    for t in range(max_new_tokens):
        logits = model.forward(idx, c)[:, -1]  # B x T x V
        probs = F.softmax(logits, dim=1)  # B x V
        new_token = torch.multinomial(probs, 1)
        idx = torch.cat([idx, new_token], dim=1)

    return idx

def batch_decode(out_tokens: Tensor) -> list[str]:
    decoded_strings = []
    for x in out_tokens:
        decoded_strings.append(decode_tokens(x.tolist(), remove_special_tokens=True))

    return decoded_strings


samples = generate(model, idx=torch.full(size=(4, 1), fill_value=1, dtype=int),
    c=torch.full(size=(4,), fill_value=1, dtype=int), max_new_tokens=40)
print('\n'.join(batch_decode(samples)))

CKTNWRYMALQIALQGRYMKHWHNHXYWMKF
WWTYHXKTQPHLELMFPRRTFSESMIAETSVC
YHPVMEMAXSPKFPLIHVSPNKCNNCLAAGRFMV
KNPTSFNFKSHKHSNWDYSLVSHLAAWPPMCLHCWM


In [17]:
from collections import defaultdict
from typing import Callable

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def training_step(
    batch: tuple[torch.Tensor, torch.Tensor],
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: str = "cpu",
) -> tuple[Tensor, Tensor]:
    model.to(device=device)
    # прогоняем батч через модель
    x, y = batch
    logits = model(x.to(device=device), y.to(device=device))
    tokens = x.to(device=device)
    # оцениваем значение ошибки
    preds = logits[:, :-1].reshape(-1, len(VOCAB))
    targets = tokens[:, 1:].reshape(-1)
    loss = F.cross_entropy(preds, targets)
    # обновляем параметры
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    # возвращаем значение функции ошибки для логирования
    return loss, preds, targets


def train_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    max_batches: int = 100,
    device: str = "cpu",
) -> dict[str, float]:
    model.train()
    loss_total = 0
    n_correct = 0
    n_total = 0
    for i, batch in enumerate(dataloader):
        x, y = batch
        loss, preds, targets = training_step(batch, model, optimizer, device)

        # save stats
        n_total += targets.size(0)
        # print('n_total += ', targets.size(0))
        n_correct += (targets.to(device=device) == preds.argmax(dim=1)).sum().item()
        # print('n_correct += ', (targets.to(device=device) == preds.argmax(dim=1)).sum())
        loss_total += targets.size(0) * loss.item()
        if i == max_batches:
            break

    return {
        "loss": loss_total / n_total,
        "accuracy": n_correct / n_total,
    }


@torch.no_grad()
def test_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    max_batches: int = 100,
    device: str = "cpu",
) -> Tensor:
    model.eval()
    model.to(device=device)
    loss_total = 0
    n_correct = 0
    n_total = 0
    for i, batch in enumerate(dataloader):
        # прогоняем батч через модель
        x, y = batch
        logits = model(x.to(device=device), y.to(device=device))
        tokens = x.to(device=device)
        # оцениваем значение ошибки
        preds = logits[:, :-1].reshape(-1, len(VOCAB))
        targets = tokens[:, 1:].reshape(-1)
        loss = F.cross_entropy(preds, targets)
        # save stats
        n_total += targets.size(0)
        n_correct += (targets.to(device=device) == preds.argmax(dim=1)).sum()
        print(targets.to(device=device), preds.argmax(dim=1))
        loss_total += targets.size(0) * loss.item()
        if i == max_batches:
            break

    return {
        "loss": loss_total / n_total,
        "accuracy": n_correct / n_total,
    }


def run_experiment(
    model_gen: Callable[[], nn.Module],
    optim_gen: Callable[[nn.Module], torch.optim.Optimizer],
    seed: int,
    n_epochs: int = 10,
    max_batches: int | None = None,
    verbose: bool = False,
    device: str = "cpu",
) -> dict[str, list[float]]:
    torch.manual_seed(seed)
    model = model_gen()
    optim = optim_gen(model)
    metrics: dict[str, list[float]] = defaultdict(list)
    for i in range(n_epochs):
        train_dict = train_epoch(
            train_loader, model, optim, max_batches=max_batches, device=device
        )
        test_dict = test_epoch(
            test_loader, model, max_batches=max_batches, device=device
        )
        train_loss, train_accuracy = train_dict["loss"], train_dict["accuracy"]
        test_loss, test_accuracy = test_dict["loss"], test_dict["accuracy"]
        if verbose:
            print(
                f"Epoch {i} train: loss = {train_loss:.4f}"
            )
            print(
                f"Epoch {i} test: loss = {test_loss:.4f}"
            )

        metrics["train_losses"].append(train_loss)
        metrics["test_losses"].append(test_loss)

    return model, metrics

In [18]:
from time import perf_counter
print("no lstm")

start=perf_counter()
model = RNNGen(vocab_size=len(VOCAB))
_, metrics = run_experiment(
    model_gen=lambda: model,
    optim_gen=lambda x: torch.optim.Adam(x.parameters(), lr=0.001),
    seed=42,
    n_epochs=10,
    max_batches=None,
    verbose=True,
    device="cuda",
)
print(f"ran {perf_counter()-start} seconds")

no lstm
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  2,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  2,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 16, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 15,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], d

In [24]:
! pip install biopython==1.84

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.3 MB/s eta 0:00:00


In [25]:
from Bio.Align import PairwiseAligner


def get_sequence_score(query_sequence: str) -> float:
    references = [
        "QVQLQQPGAELVKPGASVKMSCKAS_WITWVKQRPGQGLEWIGDI_TNYNEKFKTKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR_WGQGTSVTVSS",
        "EVKLEESGGGLVQPGGSMKLSCAAS_WMDWVRQSPEKGLEWVAEI_TYYAESVKGRFTISRDDSKSSVYLQMNSLRAEDTGIYYCTA_WGQGTLVTVSA",
        "HVQLVESGGGSVQPGGSLRLSCTVS_CMGWFRRAPGKEREKVATL_TSYADSVKGRFAISQDPAKNTLWLQMNSLKPEDSATYYCAA_SSRGTQVTVS",
        "QVQLVESGGGSVQAGGSLKLSCAAS_CMGWSRQAPGKGREGVARI_TAYADSVKGRFTISHDSIKNTLYLQMNSLRPEDTAKYYCAA_WGQGTQVTV",
        "QSLEESGGDLVKPGASLTLTCTAS_YMCWVRQAPGKGLEWIACI_TYYASWAKGRFTISKTSSTTVTLQMTSLTAADTATYFCAS_WGQGTLVTVS",
        "QEQLVESGGGLVKPGASLTLTCKAS_VMCWVRQAPGKGLEWIACI_SVYASWAKGRSTISRTSSTTVTLQMTSLTAADTATYFCAR_RGPGTLVTVS",
        "SVKVSCKAS_WIQWVRQAPGQGLEWMGWM_TTYSPKFQGRVSMTSDKSITTAYLELRGLTSDDTAVYSCAR_WGQGTLITVTS",
    ]
    max_score = 0.0
    for ref in map(lambda s: s.replace("_", ""), references):
        alignment = PairwiseAligner().align(query_sequence, ref)[0]
        n_matches = sum(
            1 for i in range(alignment.length) if alignment[0][i] == alignment[1][i]
        )
        max_score = max(max_score, n_matches / len(ref))

    return max_score

In [38]:
get_sequence_score("SVKVSCKAPEDNYSDYWGQGTGTTVTVSS")

0.3

In [45]:
for s in SPECIES.values():
    samples = generate(model, idx=torch.full(size=(20, 1), fill_value=1, dtype=int).to('cuda'),
        c=torch.full(size=(20,), fill_value=s, dtype=int).to('cuda'), max_new_tokens=40)
    for sample in samples:
        word = decode_tokens(sample.tolist(), remove_special_tokens=True)
        print(get_sequence_score(word), word)
    #print('\n'.join(batch_decode(samples)))

0.3191489361702128 HVQLVEAGGGIVQAGGGLRLSCVASEITYAYACYCMGWVR
0.35106382978723405 HVQLVESGGGSVQAGGSLTLSCAASGYGLADSVCGGTGSL
0.30851063829787234 EVKGGGTEREGVACMGNGKGNAYYADSVKGRFTISSDNSK
0.2631578947368421 QLVESGAGACKVDDRGTTVIYCMTWFRQPPGLAMGRFGAA
0.3829787234042553 HVQLVESGGGSVQAGGSVKLSCTAAGYWSRGWGQGTQVTV
0.3684210526315789 HVQLVESGGGSVQAGGSLRLSCVASGFTSFAYCMGWYTQV
0.3404255319148936 HVQLVESGGGSVQAGGRLRLSCAASGFTYNYLGQITQSKD
0.35106382978723405 QLVESGGGSVQAGGSLRLSCAASGDSFDYSDGWMKWRRQP
0.2872340425531915 CKAAGDGDAAPGSFGTITYAESVKGRFTISRDIAKNYVEM
0.3191489361702128 HVQLGESGGGIVQAGGSVEISCCATQVTARYAGSLQGTQS
0.3617021276595745 QVQLVESGGGSVQAGGSLRLSCDADGFGFAYWGQGKAVTV
0.30526315789473685 HVQLVHAGGSLEGSGGGSVQAGGSLTLSCVASISSDSAIH
0.3473684210526316 HVQLVESGGGSVQAGGSLRLSCTASAGGASNTHYVKSV
0.2736842105263158 HEQLGGSLRLPETESCMGTENIGSSEWSSHRGPGTQVPGP
0.2978723404255319 HVQLVQSGGSLKWPGDESPGVLRIHLGFTYRTTDVSTVYA
0.3157894736842105 HVQLVESGGGGVEGGGSLAESGGGLVTLSAEDGSTNYADP
0.2872340425531915 VQLVESGGDAVEWGGGVAQ

In [53]:
class ConditionalLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.linear_f = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_i = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_c = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.linear_o = nn.Linear(input_dim + hidden_dim, hidden_dim)

    """
    (x_{t}, h_{t-1}, c) -> h_{t}
    """
    def forward(self, x: Tensor, c: Tensor, h: Tensor) -> Tensor:
        # x: B x input_dim: эмбеддинг последнего токен
        # c: B x class_dim: эмбеддинг биологического вида
        # h: B x hidden_dim: последнее скрытое состояние
        h = torch.cat([x, h, c], dim=1)
        h = self.linear(h)
        return F.tanh(h)

class ConditionalLSTMCell(nn.Module):
    """
    (x_{t}, h_{t-1}) -> h_{t}
    """

    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.linear_f = nn.Linear(input_dim + hidden_dim + hidden_dim, hidden_dim)
        self.linear_i = nn.Linear(input_dim + hidden_dim + hidden_dim, hidden_dim)
        self.linear_c = nn.Linear(input_dim + hidden_dim + hidden_dim, hidden_dim)
        self.linear_o = nn.Linear(input_dim + hidden_dim + hidden_dim, hidden_dim)

    def forward(self, x: Tensor, h: Tensor, c: Tensor, C: Tensor) -> Tensor:
        f = torch.sigmoid(self.linear_f(torch.cat([x, h, c], dim=1)))
        i = torch.sigmoid(self.linear_i(torch.cat([x, h, c], dim=1)))
        C_ = F.tanh(self.linear_c(torch.cat([x, h, c], dim=1)))
        C = f * C + i * C_
        o = torch.sigmoid(self.linear_o(torch.cat([x, h, c], dim=1)))
        return o * F.tanh(C), C

import torch.nn as nn
from typing import Literal

class RNNGenLSTM(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_dim: int = 64, n_classes: int = 5
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.embed_c = nn.Embedding(n_classes, hidden_dim)
        self.init_h = nn.Parameter(data=torch.randn(1, hidden_dim))
        self.lstm = ConditionalLSTMCell(hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        self.init_C = nn.Parameter(data=torch.randn(1, hidden_dim))

    def forward(self, x: Tensor, c: Tensor) -> Tensor:
        # x: B x T
        # embed(x): B x T -> B x T x hidden_dim
        B, T = x.shape

        x = self.embed(x)  # B x T x hidden_dim
        c = self.embed_c(c).expand((B, -1))  # B x hidden_dim
        h = self.init_h.expand((B, -1)) # B x hidden_dim

        C = self.init_C.expand((B, -1))

        logits = []
        for t in range(T):
            xt = x[:, t, :] # B x hidden
            h, C = self.lstm.forward(xt, h, c, C)
            y = self.lm_head(h).unsqueeze(1)  # B x 1 x hidden
            logits.append(y)

        return torch.cat(logits, dim=1)

tokens, labels = next(iter(train_loader))
print("Tokens shape: ", tokens.shape, "\nLabels shape: ", labels.shape)
# print(tokens)
# print(labels)
model = RNNGen(
    vocab_size=len(VOCAB),
)
print("vocab", len(VOCAB))
logits = model.forward(tokens, labels)
print("Logits shape: ", logits.shape)
# torch.nn.CrossEntropyLoss()(model.forward(tokens, labels), )
# no errors, nice

Tokens shape:  torch.Size([32, 127]) 
Labels shape:  torch.Size([32])
vocab 24
Logits shape:  torch.Size([32, 127, 24])


In [54]:
from time import perf_counter
print("lstm")

start=perf_counter()
model = RNNGenLSTM(vocab_size=len(VOCAB))
_, metrics = run_experiment(
    model_gen=lambda: model,
    optim_gen=lambda x: torch.optim.Adam(x.parameters(), lr=0.001),
    seed=42,
    n_epochs=10,
    max_batches=None,
    verbose=True,
    device="cuda",
)
print(f"ran {perf_counter()-start} seconds")

lstm
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  2,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  2,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 18,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 18,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 16, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 12, 12,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 18,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16, 20, 16,  ...,  0,  0,  0], devi

In [55]:
from time import perf_counter
print("lstm +10 epochs")

start=perf_counter()
_, metrics2 = run_experiment(
    model_gen=lambda: model,
    optim_gen=lambda x: torch.optim.Adam(x.parameters(), lr=0.001),
    seed=42,
    n_epochs=10,
    max_batches=None,
    verbose=True,
    device="cuda",
)
print(f"ran {perf_counter()-start} seconds")

lstm +10 epochs
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  2,  0,  0], device='cuda:0') tensor([ 9, 20, 16,  ...,  2,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 16, 16,  ...,  0,  0,  0], device='cuda:0') tensor([16,  6, 12,  ...,  0,  0,  0], device='cuda:0')
tensor([16, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0')
tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0') tensor([18, 20, 11,  ...,  0,  0,  0], device='cuda:0')
tensor([ 9, 20, 16,  ...,  0,  0,  0], device='cuda:0') tensor([ 9, 20, 16,  ...,  0,  0

In [56]:
for s in SPECIES.values():
    samples = generate(model, idx=torch.full(size=(20, 1), fill_value=1, dtype=int).to('cuda'),
        c=torch.full(size=(20,), fill_value=s, dtype=int).to('cuda'), max_new_tokens=40)
    for sample in samples:
        word = decode_tokens(sample.tolist(), remove_special_tokens=True)
        print(get_sequence_score(word), word)
    # print('\n'.join(batch_decode(samples)))

0.2978723404255319 QVHVESGGGYLSAQGSLRLSCTASGYTRYSHSYCMGWFRQ
0.3404255319148936 VQLVESGGESVQAGGSLRLTCAASGFTHSSLEMAWFRQAP
0.35789473684210527 HVQLVESGGGSVQAGGSLRLSCVVTGDASYSMCWVRQVSP
0.3368421052631579 HVQLVESGGGIVQAGGSLRLSCAVSGYTYSYCMGWLRQRS
0.30851063829787234 SCMSGGSVYRSDLGWLRQAPGKDREGVATIASDGSTYTQP
0.35106382978723405 HVQLVESGGGSVQAGGSLRLSCVASGDTYVLSCMAWFRQA
0.3404255319148936 VQLVESGGGSVQAGGSLRLSCAVSGFTSSSSCMGWVRQAP
0.3263157894736842 HVQLVESGGGSVRPGATLRLSCVTSGYSDVRTYCMGWFRQ
0.2978723404255319 WVQLGPGERESVVAASIITTHYSDSVKGRFTISRDNANTT
0.32978723404255317 QVQLVESGGGQVQLVGSLSLSCAAGGNTFSTYCLWFRQAA
0.2875 SVESGGGSVRAPGNPTTLSCQAPELTYSTNHMAWFRQAPG
0.3473684210526316 HVQLVESGGGSVQAGGSLRLSCVASRGFIDTCMAWFRQAP
0.3157894736842105 HVQLVESGGGSVQPGASLRLASPPTTEDYCEGDTGWVRQA
0.3368421052631579 LVESGGGSVQAGGSLRLSCVASGFTDTSGCRGWLRQAPGK
0.3191489361702128 VHVQLVESGGGSVQAGGSLRLSCAATDTTTTTGYCMGWFR
0.3191489361702128 HVQLVESGGGRVQAGGSLTLSCTASGDFLSRHCMGWFRQA
0.3368421052631579 HVQLVESGGGSVQAGGSLRLSCTASGNIA